In [ ]:
import pandas as pd

from TerraFin.analytics.similarity.pool import get_pool
from TerraFin.analytics.similarity.scorer import score_pool
from TerraFin.data import get_data_factory


df = get_data_factory()

## Download

In [ ]:
pool = get_pool("sp500+nasdaq100+kospi200")
prices = pool.prices()
pool.info()

## Sanity check

In [ ]:
lengths = pd.Series({sym: len(s) for sym, s in prices.items()})
starts = pd.Series({sym: s.index[0] for sym, s in prices.items()})
ends = pd.Series({sym: s.index[-1] for sym, s in prices.items()})
nan_syms = [sym for sym, s in prices.items() if s.isna().any()]

print(f"loaded  {len(prices)}/{len(pool.symbols)}")
print(f"days    min={lengths.min():.0f}  mean={lengths.mean():.0f}  max={lengths.max():.0f}")
print(f"start   {starts.min().date()} → {starts.max().date()}")
print(f"end     {ends.min().date()} → {ends.max().date()}")
print(f"NaN     {len(nan_syms)}")
if nan_syms:
    print(nan_syms)

## Similarity search

In [ ]:
TARGET = "NVDA"
PERIOD = "6m"

chunk = df.get_recent_history(TARGET, period=PERIOD)
target = chunk.frame.set_index("time")["close"].ffill().dropna()

candidates = {sym: s for sym, s in pool.prices().items() if sym != TARGET}
results = score_pool(target, candidates, names=pool.names(), top_n=10)

pd.DataFrame(
    [
        {
            "symbol": r.symbol,
            "name": r.name,
            "score": r.score,
            "match_start": r.match_start,
            "match_end": r.match_end,
            "days": r.overlap_days,
        }
        for r in results
    ]
)

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np


AFTER_DAYS = 21  # ~1 month of trading days

# Korean font support (macOS)
_cjk = next(
    (
        f.name
        for f in fm.fontManager.ttflist
        if "AppleGothic" in f.name or "Malgun" in f.name or "NanumGothic" in f.name
    ),
    None,
)
if _cjk:
    plt.rcParams["font.family"] = _cjk


def cumlog_pct(s):
    v = s.values.astype(float)
    v = np.where(v <= 0, np.nan, v)
    return np.nan_to_num(np.log(v / v[0]) * 100)


target_cl = cumlog_pct(target)
n = len(target_cl)

fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    f"Top-{len(results)} matches for {TARGET} ({PERIOD})\nBlue=target  Orange=match  Orange-dashed=after 1m",
    fontsize=12,
)
gs = gridspec.GridSpec(5, 2, figure=fig, hspace=0.55, wspace=0.3)

for i, r in enumerate(results):
    ax = fig.add_subplot(gs[i // 2, i % 2])

    series = pool.prices()[r.symbol]
    ms = pd.Timestamp(r.match_start)
    me = pd.Timestamp(r.match_end)

    window = series[(series.index >= ms) & (series.index <= me)]
    after = series[series.index > me].iloc[:AFTER_DAYS]

    win_cl = cumlog_pct(window)
    # after_cl: incremental gain/loss from window end — must add win_cl[-1] to continue on same axis
    after_cl = np.nan_to_num(np.log(after.values / window.values[-1]) * 100) if len(after) else np.array([])

    x_target = np.arange(n)
    x_win = np.arange(len(win_cl))
    x_after = np.arange(len(win_cl) - 1, len(win_cl) + len(after_cl))
    after_y = np.concatenate([[win_cl[-1]], win_cl[-1] + after_cl]) if len(after_cl) else np.array([])

    ax.axvline(x=n - 1, color="grey", lw=0.8, ls=":")
    ax.axhline(0, color="grey", lw=0.5)
    ax.plot(x_target, target_cl, color="steelblue", lw=1.2, label="target")
    ax.plot(x_win, win_cl, color="darkorange", lw=1.2, label="match")
    if len(after_y):
        ax.plot(x_after, after_y, color="darkorange", lw=1.2, ls="--", label="after")

    name_str = f" ({r.name})" if r.name else ""
    ax.set_title(f"{r.symbol}{name_str}  score={r.score:.3f}\n{r.match_start} → {r.match_end}", fontsize=8)
    ax.set_ylabel("%", fontsize=7)
    ax.tick_params(labelsize=7)

plt.show()